In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import camb
import ipywidgets as widgets
from ipywidgets import interact

# --- 1. MODELO DE REFERÊNCIA (Calculado uma vez) ---
pars_base = camb.CAMBparams()
pars_base.set_cosmology(H0=67.4, ombh2=0.0224, omch2=0.120, omk=0.0)
pars_base.InitPower.set_params(As=2.1e-9, ns=0.965)
pars_base.set_for_lmax(2500, lens_potential_accuracy=0)

resultados_base = camb.get_results(pars_base)
cl_base = resultados_base.get_cmb_power_spectra(pars_base, CMB_unit='muK')['total'][:, 0]
ls_base = np.arange(cl_base.shape[0])

# --- 2. FUNÇÃO E DECORADOR (A forma mais robusta no Jupyter) ---
# O @interact cria a interface automaticamente acima da função
@interact(
    H0=widgets.FloatSlider(value=67.4, min=50.0, max=85.0, step=0.5, description='H0:', continuous_update=False),
    ombh2=widgets.FloatSlider(value=0.0224, min=0.010, max=0.040, step=0.001, description='Ωb h²:', continuous_update=False),
    omch2=widgets.FloatSlider(value=0.120, min=0.050, max=0.250, step=0.005, description='Ωc h²:', continuous_update=False),
    omk=widgets.FloatSlider(value=0.0, min=-0.05, max=0.05, step=0.01, description='Ωk:', continuous_update=False),
    ns=widgets.FloatSlider(value=0.965, min=0.85, max=1.10, step=0.01, description='ns:', continuous_update=False),
    As_mult=widgets.FloatSlider(value=1.0, min=0.5, max=1.5, step=0.05, description='Amplitude:', continuous_update=False),
    escala_log=widgets.Checkbox(value=True, description='Escala Logarítmica')
)
def plot_espectro(H0, ombh2, omch2, omk, ns, As_mult, escala_log):
    # Calcula a nova cosmologia via CAMB
    pars = camb.CAMBparams()
    pars.set_cosmology(H0=H0, ombh2=ombh2, omch2=omch2, omk=omk)
    pars.InitPower.set_params(As=2.1e-9 * As_mult, ns=ns)
    pars.set_for_lmax(2500, lens_potential_accuracy=0)
    
    resultados = camb.get_results(pars)
    cl_custom = resultados.get_cmb_power_spectra(pars, CMB_unit='muK')['total'][:, 0]
    
    # Desenha o gráfico
    fig, ax = plt.subplots(figsize=(10, 5))
    plt.style.use('dark_background')
    
    ax.plot(ls_base[2:2501], cl_base[2:2501], color='#94a3b8', linestyle='--', linewidth=2, label='ΛCDM (Planck 2018)')
    ax.plot(ls_base[2:2501], cl_custom[2:2501], color='#ef4444', linewidth=2.5, label='Modelo Modificado')
    
    ax.set_xlabel(r'Multipolo $\ell$', fontsize=12)
    ax.set_ylabel(r'$\ell(\ell+1)C_\ell / 2\pi \quad [\mu K^2]$', fontsize=12)
    
    if escala_log:
        ax.set_xscale('log')
    
    ax.set_xlim(2, 2500)
    ax.set_ylim(0, 7500)
    ax.grid(color='#334155', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.legend(loc='upper right')
    
    plt.show()
    plt.close(fig)